# EDA - Bergen bysykkel (Del 1)
Mål: bygge timesrutenett per målstasjon, bruke LOCF for `free_bikes`, lage label `y_t_plus_1h`, merge vær + trips, features, kvalitetsjekker, split (70/15/15).

### Imports

In [28]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split

ROOT = Path.cwd()
while not (ROOT / "raw_data").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
RAW = ROOT / "raw_data"
OUT = ROOT / "data" / "processed"
FIG = ROOT / "reports" / "figures"
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

TARGETS = [
    "Møllendalsplass","Torgallmenningen","Grieghallen","Høyteknologisenteret",
    "Studentboligene","Akvariet","Damsgårdsveien 71","Dreggsallmenningen Sør",
    "Florida Bybanestopp"
]

print("ROOT =", ROOT)

ROOT = c:\Users\adria\Downloads\DataScience\H25\INF161\Prosjekt1


### Hjelpefunksjoner

In [29]:
def parse_ts(series: pd.Series) -> pd.Series:
    """Parser for tid med/uten mikrosekunder og med tz."""
    s = pd.to_datetime(series, format="%Y-%m-%d %H:%M:%S.%f%z", utc=True, errors="coerce")
    m = s.isna()
    if m.any():
        s.loc[m] = pd.to_datetime(series[m], format="%Y-%m-%d %H:%M:%S%z", utc=True, errors="coerce")
    return s

def hourly_fb_for_station(stations_df: pd.DataFrame, station_name: str) -> pd.DataFrame:
    """LOCF for stasjoner"""
    g = (stations_df[stations_df["station"] == station_name]
         .sort_values("timestamp")[["timestamp","free_bikes"]])
    if g.empty:
        return pd.DataFrame(columns=["station","time","free_bikes"])
    hrs = pd.DataFrame({"time": pd.date_range(g["timestamp"].min().floor("h"),
                                              g["timestamp"].max().ceil("h"),
                                              freq="h", tz="UTC")})
    merged = pd.merge_asof(hrs, g.rename(columns={"timestamp":"time"}),
                           on="time", direction="backward")
    merged["station"] = station_name
    return merged[["station","time","free_bikes"]]

def to_utc(ts):
    """Gjør grenseverdier oppmerksom på tz (UTC) for slicing."""
    return None if ts is None else pd.to_datetime(ts, utc=True)

### Leser inn data

In [30]:
# Stations
stations = pd.read_csv(RAW / "stations.csv")
stations["timestamp"] = pd.to_datetime(stations["timestamp"], utc=True)

# Trips
trips = pd.read_csv(RAW / "trips.csv")
trips["started_at"] = parse_ts(trips["started_at"])
trips["ended_at"]   = parse_ts(trips["ended_at"])

# Weather
weather = pd.read_csv(RAW / "weather.csv")
weather["timestamp"] = pd.to_datetime(weather["timestamp"], utc=True)

# Filtrer til målstasjoner i stations
stations_t = stations[stations["station"].isin(TARGETS)].copy()
print("Unike målstasjoner funnet:", stations_t["station"].nunique(), "av", len(TARGETS))

Unike målstasjoner funnet: 9 av 9


### Datasettet har få datapunkter for perioden april 2024 til august 2024. Vi ekskluderer denne perioden fra dataen vår før vi går videre

In [31]:
BAD_START = pd.Timestamp("2024-04-01 00:00:00", tz="UTC")
BAD_END   = pd.Timestamp("2024-08-31 23:59:59", tz="UTC")

stations = stations[~((stations["timestamp"] >= BAD_START) & (stations["timestamp"] <= BAD_END))].copy()

trips = trips[~(
    ((trips["started_at"] >= BAD_START) & (trips["started_at"] <= BAD_END)) |
    ((trips["ended_at"]   >= BAD_START) & (trips["ended_at"]   <= BAD_END))
)].copy()

weather = weather[~((weather["timestamp"] >= BAD_START) & (weather["timestamp"] <= BAD_END))].copy()

### LOCF + label - hourly_fb

In [32]:
# LOCF per stasjon til timesrutenett
hourly_parts = [hourly_fb_for_station(stations_t, s) for s in TARGETS]
hourly_fb = pd.concat(hourly_parts, ignore_index=True).sort_values(["station","time"])

# Label: y_t_plus_1h
hourly_fb = hourly_fb.sort_values(["station","time"]).reset_index(drop=True)
t_next = hourly_fb.groupby("station")["time"].shift(-1)
y_next = hourly_fb.groupby("station")["free_bikes"].shift(-1)
valid = (t_next - hourly_fb["time"] == pd.Timedelta(hours=1))
hourly_fb["y_t_plus_1h"] = np.where(valid, y_next, np.nan)
hourly_fb = hourly_fb.dropna(subset=["free_bikes","y_t_plus_1h"]).reset_index(drop=True)

display(hourly_fb.head(3))

,station,time,free_bikes,y_t_plus_1h
0,Akvariet,2024-04-10 22:00:00+00:00,0.0,0.0
1,Akvariet,2024-04-10 23:00:00+00:00,0.0,0.0
2,Akvariet,2024-04-11 00:00:00+00:00,0.0,0.0


### LOCF visualisering

In [33]:
def plot_obs_vs_locf(hourly_like_df, station, start=None, end=None, freq="D"):
    """Observasjon y(t+1) vs. baseline y(t) for én stasjon."""
    s = (hourly_like_df[hourly_like_df["station"] == station]
         .sort_values("time")[["time","free_bikes","y_t_plus_1h"]]
         .rename(columns={"free_bikes":"y_pred"}))
    s = s[(s["time"] >= to_utc(start))] if start else s
    s = s[(s["time"] <= to_utc(end))]   if end   else s
    s = s.set_index("time").resample(freq).median().reset_index()
    fig = px.line(s, x="time", y=["y_t_plus_1h","y_pred"],
                  title=f"{station} - Observasjon vs. LOCF (freq={freq})")
    fig.update_layout(xaxis_title="", yaxis_title="free_bikes", legend_title="")
    fig.show()
    
    #fig.write_image(str(FIG / "LOCF_vis.png"), scale=2, width=1100, height=600)

plot_obs_vs_locf(hourly_fb, "Torgallmenningen", start="2025-04-01", end="2025-04-07", freq="1H")

C:\Users\adria\AppData\Local\Temp\ipykernel_6436\2409757345.py:8: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.



### Vær på timesnivå og tidsfeatures i df

In [34]:
w_hourly = (weather.sort_values("timestamp")
                   .set_index("timestamp")
                   .resample("h")
                   .agg({"temperature":"mean","precipitation":"sum","wind_speed":"mean"})
                   .reset_index()
                   .rename(columns={"timestamp":"time"}))

df = hourly_fb.merge(w_hourly, on="time", how="left")

# Tidsfeatures
df["hour_of_day"] = df["time"].dt.hour
df["day_of_week"] = df["time"].dt.dayofweek
df["is_weekend"]  = (df["day_of_week"] >= 5)

display(df.head(3))

,station,time,free_bikes,y_t_plus_1h,temperature,precipitation,wind_speed,hour_of_day,day_of_week,is_weekend
0,Akvariet,2024-04-10 22:00:00+00:00,0.0,0.0,NaN,0.0,NaN,22,2,False
1,Akvariet,2024-04-10 23:00:00+00:00,0.0,0.0,NaN,0.0,NaN,23,2,False
2,Akvariet,2024-04-11 00:00:00+00:00,0.0,0.0,NaN,0.0,NaN,0,3,False


### Trips til departures og arrivals og net_flow

In [35]:
trips["hour"] = trips["started_at"].dt.floor("h")

deps = (trips[trips["start_station_name"].isin(TARGETS)]
        .groupby(["start_station_name","hour"]).size()
        .rename("departures").reset_index()
        .rename(columns={"start_station_name":"station","hour":"time"}))

arrs = (trips[trips["end_station_name"].isin(TARGETS)]
        .groupby(["end_station_name","hour"]).size()
        .rename("arrivals").reset_index()
        .rename(columns={"end_station_name":"station","hour":"time"}))

df = (df.merge(deps, on=["station","time"], how="left")
        .merge(arrs, on=["station","time"], how="left"))
df[["departures","arrivals"]] = df[["departures","arrivals"]].fillna(0).astype(int)

# Rullende 3t-summer (ingen lekkasje - kun mindre eller lik t)
df = df.sort_values(["station","time"])
tmp = df.set_index("time")

for col in ["departures","arrivals"]:
    tmp[f"{col}_3h"] = (tmp.groupby("station")[col]
                          .rolling("3H", min_periods=1).sum()
                          .reset_index(level=0, drop=True))

df = tmp.reset_index()

# Netto flyt
df["net_flow"]    = df["arrivals"]    - df["departures"]
df["net_flow_3h"] = df["arrivals_3h"] - df["departures_3h"]

display(df.head(3))

C:\Users\adria\AppData\Local\Temp\ipykernel_6436\760288800.py:23: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.

C:\Users\adria\AppData\Local\Temp\ipykernel_6436\760288800.py:23: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.



,time,station,free_bikes,y_t_plus_1h,temperature,precipitation,wind_speed,hour_of_day,day_of_week,is_weekend,departures,arrivals,departures_3h,arrivals_3h,net_flow,net_flow_3h
0,2024-04-10 22:00:00+00:00,Akvariet,0.0,0.0,NaN,0.0,NaN,22,2,False,0,0,0.0,0.0,0,0.0
1,2024-04-10 23:00:00+00:00,Akvariet,0.0,0.0,NaN,0.0,NaN,23,2,False,0,0,0.0,0.0,0,0.0
2,2024-04-11 00:00:00+00:00,Akvariet,0.0,0.0,NaN,0.0,NaN,0,3,False,0,0,0.0,0.0,0,0.0


In [36]:
# Andel ledige sykler (p_free_bikes)
df_pf = df.copy()
if "capacity" in df_pf.columns:
    cap = df_pf["capacity"]
elif "free_spots" in df_pf.columns:
    cap = df_pf["free_bikes"] + df_pf["free_spots"]
else:
    # Proxy-kapasitet, maks observert free_bikes per stasjon
    cap = df_pf.groupby("station")["free_bikes"].transform("max")

df_pf["p_free_bikes"] = (df_pf["free_bikes"] / cap).clip(0, 1)

# Fjerner perioden med dårlig dekning (samme som i modelleringen)
BAD_START = pd.Timestamp("2024-04-01 00:00:00", tz="UTC")
BAD_END   = pd.Timestamp("2024-08-31 23:59:59", tz="UTC")
df_pf = df_pf.loc[~((df_pf["time"] >= BAD_START) & (df_pf["time"] <= BAD_END))].copy()

## Helsesjekk

In [37]:
df = df.sort_values(["station","time"]).reset_index(drop=True)

# Unikhet
dups = df.duplicated(["station","time"]).sum()
assert dups == 0, f"Fant {dups} duplikate rader i (station,time)"

# Label finnes
assert df["y_t_plus_1h"].notna().all(), "Label har NaN"

# Label == y_t_plus_1h
chk  = df.groupby("station")["free_bikes"].shift(-1)
mask = df.groupby("station")["time"].shift(-1).notna()
assert (df.loc[mask, "y_t_plus_1h"].values == chk.loc[mask].values).all(), "Label ikke lik next-hour free_bikes"

# Verdier
assert (df["free_bikes"] >= 0).all(), "free_bikes < 0"

# Stasjonsoversikt
check = (
    df.groupby("station").agg(
        n=("free_bikes", "size"),
        zero_rate=("free_bikes", lambda s: float((s == 0).mean())),
        min_fb=("free_bikes", "min"),
        max_fb=("free_bikes", "max"),
    )
    .reset_index()
    .sort_values("zero_rate")
)
display(check)

,station,n,zero_rate,min_fb,max_fb
1,Damsgårdsveien 71,9282,0.083926,0.0,19.0
6,Møllendalsplass,9282,0.104611,0.0,17.0
7,Studentboligene,9282,0.112907,0.0,31.0
4,Grieghallen,9282,0.118078,0.0,16.0
2,Dreggsallmenningen Sør,9282,0.122926,0.0,34.0
8,Torgallmenningen,9282,0.130144,0.0,24.0
3,Florida Bybanestopp,9282,0.226783,0.0,25.0
0,Akvariet,9281,0.300614,0.0,18.0
5,Høyteknologisenteret,9282,0.617970,0.0,25.0


### Laggs, model_df og lagring av forhåndsversjon

In [38]:
# Lagg-features
for k in [1,2,3]:
    df[f"free_bikes_lag{k}"] = df.groupby("station")["free_bikes"].shift(k)
df["free_bikes_ma3"] = (df.groupby("station")["free_bikes"]
                          .rolling(3, min_periods=1).mean()
                          .reset_index(level=0, drop=True))

# Modellklart datasett
model_df = df.dropna(subset=["y_t_plus_1h","free_bikes_lag1"]).copy()

# Fjerner dårlig periode og bevarer bare rader der neste time finnes
BAD_START = pd.Timestamp("2024-04-01 00:00:00", tz="UTC")
BAD_END   = pd.Timestamp("2024-08-31 23:59:59", tz="UTC")

model_df = model_df.loc[~((model_df["time"] >= BAD_START) & (model_df["time"] <= BAD_END))].copy()
model_df = model_df.sort_values(["station","time"]).reset_index(drop=True)

next_time = model_df.groupby("station")["time"].shift(-1)
mask_ok   = (next_time - model_df["time"] == pd.Timedelta(hours=1))
model_df  = model_df.loc[mask_ok].copy()

print("Etter filtrering:", model_df["time"].min(), "til", model_df["time"].max(), "| rader:", len(model_df))
display(model_df.head(3))

# Lagre forhåndsversjon
out_path = OUT / "model_ready_preview.csv"
model_df.to_csv(out_path, index=False)
print("Skrevet:", out_path, "antall rader:", len(model_df))

Etter filtrering: 2024-09-01 00:00:00+00:00 til 2025-05-02 14:00:00+00:00 | rader: 52622


,time,station,free_bikes,y_t_plus_1h,temperature,precipitation,wind_speed,hour_of_day,day_of_week,is_weekend,departures,arrivals,departures_3h,arrivals_3h,net_flow,net_flow_3h,free_bikes_lag1,free_bikes_lag2,free_bikes_lag3,free_bikes_ma3
0,2024-09-01 00:00:00+00:00,Akvariet,0.0,0.0,9.4,0.0,5.2,0,6,True,0,0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0
1,2024-09-01 01:00:00+00:00,Akvariet,0.0,0.0,7.8,0.0,2.9,1,6,True,0,0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0
2,2024-09-01 02:00:00+00:00,Akvariet,0.0,0.0,7.1,0.0,4.0,2,6,True,0,0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0


Skrevet: c:\Users\adria\Downloads\DataScience\H25\INF161\Prosjekt1\data\processed\model_ready_preview.csv antall rader: 52622


### 70/15/15 split tilfeldig fordelt, med sklearn

In [39]:
LABEL = "y_t_plus_1h"

xy = model_df.dropna(subset=[LABEL]).sort_values("time").copy()

X = xy.drop(columns=[LABEL])
y = xy[LABEL]

X_train, X_valtest, y_train, y_valtest = train_test_split(
    X, y, train_size=0.7, shuffle=False, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_valtest, y_valtest, train_size=0.5, shuffle=False, random_state=42
)

train_df = X_train.copy(); train_df[LABEL] = y_train
val_df = X_val.copy(); val_df[LABEL] = y_val
test_df = X_test.copy(); test_df[LABEL] = y_test

model_df = model_df.copy()
model_df["split"] = "train"
model_df.loc[val_df.index, "split"] = "val"
model_df.loc[test_df.index, "split"] = "test"

# her kan vi bestemme hvilken data vi vil analysere, og vi ser bare på treningsdata i dette tilfellet
EDA_SCOPE = "train"  # "train", "val", "test", "all"
df_eda = {"train": train_df, "val": val_df, "test": test_df, "all": model_df}[EDA_SCOPE]

print("Størrelser:", len(train_df), len(val_df), len(test_df))
print("Stasjoner per split (train):")
print(train_df["station"].value_counts().sort_index())
print("Stasjoner per split (val):")
print(val_df["station"].value_counts().sort_index())
print("Stasjoner per split (test):")
print(test_df["station"].value_counts().sort_index())


Størrelser: 36835 7893 7894
Stasjoner per split (train):
station
Akvariet                  4093
Damsgårdsveien 71         4093
Dreggsallmenningen Sør    4093
Florida Bybanestopp       4093
Grieghallen               4092
Høyteknologisenteret      4092
Møllendalsplass           4093
Studentboligene           4093
Torgallmenningen          4093
Name: count, dtype: int64
Stasjoner per split (val):
station
Akvariet                  876
Damsgårdsveien 71         877
Dreggsallmenningen Sør    877
Florida Bybanestopp       877
Grieghallen               877
Høyteknologisenteret      878
Møllendalsplass           877
Studentboligene           877
Torgallmenningen          877
Name: count, dtype: int64
Stasjoner per split (test):
station
Akvariet                  877
Damsgårdsveien 71         877
Dreggsallmenningen Sør    877
Florida Bybanestopp       877
Grieghallen               878
Høyteknologisenteret      877
Møllendalsplass           877
Studentboligene           877
Torgallmenningen       

In [40]:
# beskrivende statistikk på treningsdata
desc_train = train_df.select_dtypes(include="number").describe().T.round(3); display(desc_train)

,count,mean,std,min,25%,50%,75%,max
free_bikes,36835.0,5.490,6.914,0.0,0.000,2.000,9.000,34.0
temperature,36421.0,5.353,5.977,-12.6,1.700,5.400,9.400,27.9
precipitation,36835.0,0.334,0.743,0.0,0.000,0.000,0.300,6.8
wind_speed,36466.0,11.267,7.938,0.0,5.000,9.200,15.900,40.6
hour_of_day,36835.0,11.483,6.922,0.0,5.000,11.000,17.000,23.0
day_of_week,36835.0,2.994,2.015,0.0,1.000,3.000,5.000,6.0
departures,36835.0,0.636,1.337,0.0,0.000,0.000,1.000,17.0
arrivals,36835.0,0.666,1.407,0.0,0.000,0.000,1.000,18.0
departures_3h,36835.0,1.908,3.182,0.0,0.000,1.000,3.000,30.0
arrivals_3h,36835.0,1.997,3.398,0.0,0.000,1.000,3.000,33.0


In [41]:
# manglende data
missing_train = train_df.isna().mean().sort_values(ascending=False).to_frame("missing_rate").round(4); display(missing_train.head(12))

weather_cols = [c for c in ["temperature","wind_speed","precipitation"] if c in train_df]
print("Vær NaN-rate:", train_df[weather_cols].isna().mean().round(4).to_dict())

,missing_rate
temperature,0.0112
wind_speed,0.0100
station,0.0000
time,0.0000
free_bikes,0.0000
precipitation,0.0000
hour_of_day,0.0000
day_of_week,0.0000
is_weekend,0.0000
departures,0.0000


Vær NaN-rate: {'temperature': 0.0112, 'wind_speed': 0.01, 'precipitation': 0.0}


## Figurer på treningsdata

In [42]:
g = (df_eda.groupby("time", as_index=False)
       .agg(total_departures=("departures","sum")))
g["weekday"] = g["time"].dt.day_name()
order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]

bar = (g.groupby("weekday", as_index=False)
        .agg(mean_dep=("total_departures","mean")))
fig = px.bar(bar, x="weekday", y="mean_dep",
             category_orders={"weekday": order},
             title="Gjennomsnittlige totale avganger per ukedag")
fig.update_layout(xaxis_title="Ukedag", yaxis_title="Avganger per time (snitt)")
fig.show()

#fig.write_image(str(FIG / "mean_totale_avganger_pr_ukedag.png"), scale=2, width=1100, height=600)

In [43]:
fig = px.box(g, x="weekday", y="total_departures",
             category_orders={"weekday": order},
             points=False,
             title="Fordeling av totale avganger per ukedag")
fig.update_layout(xaxis_title="Ukedag", yaxis_title="Avganger per time")
fig.show()

#fig.write_image(str(FIG / "Variasjon_totale_avganger_pr_ukedag.png"), scale=2, width=1100, height=600)

In [44]:
g["hour"] = g["time"].dt.hour
heat = (g.groupby(["weekday","hour"], as_index=False)
         .agg(mean_dep=("total_departures","mean")))
heat["weekday"] = pd.Categorical(heat["weekday"], categories=order, ordered=True)

pivot = heat.pivot(index="weekday", columns="hour", values="mean_dep").loc[order]
fig = px.imshow(pivot, aspect="auto", origin="lower",
                title="Totale avganger: ukedag og time (gjennomsnitt)")
fig.update_xaxes(title="Time på døgnet")
fig.update_yaxes(title="Ukedag")
fig.show()

#fig.write_image(str(FIG / "Heatmap_totale_avganger_ukedag_time.png"), scale=2, width=1100, height=600)

In [45]:
# Daglig median
daily = (df_pf.set_index("time")
              .groupby("station")["p_free_bikes"]
              .resample("D").median()
              .reset_index())

fig = px.line(
    daily, x="time", y="p_free_bikes",
    facet_col="station", facet_col_wrap=3,
    title="Andel ledige sykler per stasjon",
)
fig.update_yaxes(range=[0,1])
fig.show()


In [46]:
fig = px.histogram(
    df_eda,
    x="hour_of_day",
    y="free_bikes",
    histfunc="avg",
    facet_col="station",
    facet_col_wrap=3,
    category_orders={"hour_of_day": list(range(24))},
    title="Gjennomsnittlig ledige sykler per time, per stasjon",
    labels={"hour_of_day":"Time på døgnet", "free_bikes":"Gj.snitt ledige sykler"},
)
fig.update_traces(marker_line_width=0)
fig.update_layout(bargap=0.05, showlegend=False, height=900, width=1100)
fig.show()

#fig.write_image(str(FIG / "Mean_ledige_sykler_pr_time_pr_stasjon.png"), scale=2, width=1100, height=600)

In [47]:
metrics = ["free_bikes", "departures", "arrivals", "net_flow"]
use_metrics = [m for m in metrics if m in df_eda.columns]

plot_df = df_eda.melt(
    id_vars=["station", "hour_of_day"],
    value_vars=use_metrics,
    var_name="metric",
    value_name="value",
)

fig = px.histogram(
    plot_df,
    x="hour_of_day", y="value", histfunc="avg",
    color="metric",
    facet_col="station", facet_col_wrap=3,
    category_orders={"hour_of_day": list(range(24))},
    title="Gj.snitt per time på døgnet - flere mål per stasjon",
    labels={"hour_of_day":"Time på døgnet", "value":"Verdi", "metric":"Måling"},
)
fig.update_layout(barmode="group", bargap=0.02, showlegend=True, height=950, width=1200)
fig.update_traces(marker_line_width=0)
fig.show()

#fig.write_image(str(FIG / "Mean_per_time_ekstra.png"), scale=2, width=1100, height=600)

In [48]:
tmp = df_eda[["time", "station", "departures", "temperature"]].copy()
tmp["departures"] = tmp["departures"].fillna(0)

g2 = (tmp.groupby("time", as_index=False)
          .agg(departures=("departures", "sum"),
               temperature=("temperature", "mean"))
          .dropna(subset=["temperature"]))

fig = px.scatter(
    g2, x="temperature", y="departures",
    opacity=0.5,
    title="Totale avganger per time vs. temperatur"
)
fig.update_layout(
    xaxis_title="Temperatur",
    yaxis_title="Avganger per time (sum over stasjoner)"
)
fig.show()

#fig.write_image(str(FIG / "Totale_avganger_pr_time_vs_temp.png"), scale=2, width=1100, height=600)

In [49]:
x = g2["temperature"].to_numpy()
y = g2["departures"].to_numpy()
m, b = np.polyfit(x, np.log1p(y), 1)
x_line = np.linspace(x.min(), x.max(), 200)
y_line = np.expm1(m * x_line + b)

fig = px.scatter(g2, x="temperature", y="departures",
                 opacity=0.25,
                 title="Totale avganger vs. temperatur - eksponentiell trend")
fig.update_traces(marker=dict(size=4))
fig.add_scatter(x=x_line, y=y_line, mode="lines", name="Exp-trend (log1p)",
                line=dict(width=4))
fig.update_layout(xaxis_title="Temperatur", yaxis_title="Avganger per time (sum)")
fig.show()

#fig.write_image(str(FIG / "Totale_avganger_pr_time_vs_temp_eksp.png"), scale=2, width=1100, height=600)

In [50]:
prec = df_eda.dropna(subset=["precipitation"]).copy()
bins  = [0, 0.1, 0.5, 1, 2, 5, 10, np.inf]
labs  = ["0-0.1","0.1-0.5","0.5-1","1-2","2-5","5-10",">10"]
prec["precip_bin"] = pd.cut(prec["precipitation"], bins=bins, labels=labs, right=False)

tot_dep = (prec.groupby("precip_bin", as_index=False)
              .agg(mean_dep=("departures","mean")))
fig = px.bar(tot_dep, x="precip_bin", y="mean_dep",
       title="Totale avganger vs. nedbør (gj.snitt per time)")
fig.show()
#fig.write_image(str(FIG / "Totale_avganger_vs_nedbor.png"), scale=2, width=1100, height=600)

C:\Users\adria\AppData\Local\Temp\ipykernel_6436\908892003.py:6: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [51]:
agg_hod = (
    df_eda.groupby(["station", "hour_of_day"], as_index=False)["free_bikes"]
      .median()
      .rename(columns={"free_bikes": "median_free_bikes"})
)

fig = px.line(
    agg_hod, x="hour_of_day", y="median_free_bikes",
    color="station", markers=True,
    title="Median ledige sykler per time på døgnet (per stasjon)"
)
fig.update_layout(
    xaxis_title="Time på døgnet",
    yaxis_title="Median ledige sykler",
    legend_title="Stasjon"
)
fig.update_xaxes(dtick=1)
fig.show()

#fig.write_image(str(FIG / "Median_ledige_sykler_pr_time_pr_stasjon.png"), scale=2, width=1100, height=600)

In [52]:
baseline = (df_eda.groupby(["station","hour_of_day"])["free_bikes"]
              .median().rename("baseline").reset_index())

df_hod = df_eda.merge(baseline, on=["station","hour_of_day"], how="left")
df_hod["dev_from_typical"] = df_hod["free_bikes"] - df_hod["baseline"]

prec = df_hod.dropna(subset=["precipitation"]).copy()
edges  = [0, 0.1, 0.5, 1, 2, 5, 10, np.inf]
labels = ["0-0.1","0.1-0.5","0.5-1","1-2","2-5","5-10",">10"]
prec["precip_bin"] = pd.cut(prec["precipitation"], bins=edges, labels=labels, right=False)

grp = (prec.groupby(["station","precip_bin"])["dev_from_typical"]
          .agg(median_dev="median",
               q25=lambda s: s.quantile(0.25),
               q75=lambda s: s.quantile(0.75),
               n="size").reset_index())
grp["err"] = (grp["q75"] - grp["q25"]) / 2

fig_err = px.line(
    grp, x="precip_bin", y="median_dev", color="station", markers=True,
    error_y="err", category_orders={"precip_bin": list(grp["precip_bin"].cat.categories)},
    title="Avvik fra typisk time mot nedbør (IQR/2 som feilstolper)"
)
fig_err.update_layout(xaxis_title="Nedbør (mm/time, binned)", yaxis_title="Ledige sykler - typisk for timen")
fig_err.add_hline(y=0, line_width=1)
fig_err.show()

#fig.write_image(str(FIG / "Avvik_typisk_time_vs_nedbor_IQR_2.png"), scale=2, width=1100, height=600)

C:\Users\adria\AppData\Local\Temp\ipykernel_6436\3107736281.py:12: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [53]:
df_eda = df_eda.copy()
df_eda["delta_next"] = df_eda["y_t_plus_1h"] - df_eda["free_bikes"]
df_eda["net_flow_i"] = df_eda["net_flow"].astype(int).clip(-20, 20)

fig_sc = px.scatter(
    df_eda, x="net_flow", y="delta_next", color="station", opacity=0.5,
    title="Netto flyt (t) vs. endring i ledige sykler (t til t+1)"
)
fig_sc.update_layout(xaxis_title="Netto flyt i time t (ankomster - avganger)",
                     yaxis_title="delta ledige sykler (t til t+1)")
fig_sc.add_hline(y=0, line_width=1); fig_sc.add_vline(x=0, line_width=1)
fig_sc.show()

agg = (df_eda.groupby(["station","net_flow_i"], as_index=False)["delta_next"]
         .median().rename(columns={"delta_next":"median_delta"}))

fig_ln = px.line(
    agg, x="net_flow_i", y="median_delta", color="station", markers=True,
    title="Median delta ledige sykler per (heltalls) nettoflyt"
)
fig_ln.update_layout(xaxis_title="Netto flyt i time t", yaxis_title="Median delta ledige sykler (t til t+1)")
fig_ln.add_hline(y=0, line_width=1); fig_ln.add_vline(x=0, line_width=1)
fig_ln.show()

#fig_sc.write_image(str(FIG / "Netto_flyt_vs_endring_ledige_sykler.png"), scale=2, width=1100, height=600)
#fig_ln.write_image(str(FIG / "Median_delta_ledige_sykler_pr_nettoflyt.png"), scale=2, width=1100, height=600)

In [54]:
# Verdier til rapporten
START = model_df["time"].min()
SLUTT = model_df["time"].max()
N_TIMER = model_df["time"].nunique()

N_RADER_TOTAL = len(model_df)
N_RADER_TRAIN = len(train_df)
N_RADER_VAL = len(val_df)
N_RADER_TEST = len(test_df)

print("START:", START)
print("SLUTT:", SLUTT)
print("N_TIMER:", N_TIMER)
print("Rader total/train/val/test:", N_RADER_TOTAL, N_RADER_TRAIN, N_RADER_VAL, N_RADER_TEST)

START: 2024-09-01 00:00:00+00:00
SLUTT: 2025-05-02 14:00:00+00:00
N_TIMER: 5847
Rader total/train/val/test: 52622 36835 7893 7894
